# 6TH : Linear classification

In [8]:
import pickle
import numpy as np
import os
import matplotlib.pyplot as plt

In [9]:
def load_cifar10_data(file_path):
    with open(file_path, 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    images = batch[b'data']
    labels = batch[b'labels']
    return images, np.array(labels)

data_dir = 'cifar-10-batches-py'

all_train_images = []
all_train_labels = []

for i in range(1, 6):  # data_batch_1 ~ data_batch_5
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    images, labels = load_cifar10_data(batch_path)
    all_train_images.append(images)
    all_train_labels.append(labels)

# load whole train dataset
all_train_images = np.concatenate(all_train_images, axis=0)
all_train_labels = np.concatenate(all_train_labels, axis=0)

# load whole test dataset
test_path = os.path.join(data_dir, 'test_batch')
all_test_images, all_test_labels = load_cifar10_data(test_path)

print(f"Train dataset: {all_train_images.shape}")
print(f"Test dataset: {all_test_images.shape}")

Train dataset: (50000, 3072)
Test dataset: (10000, 3072)


In [12]:
class LinearClassifier:
    def __init__(self, input_dim, num_classes):
        # Initialize weights and bias randomly

        self.W = np.random.randn(input_dim, num_classes)
        self.b = np.zeros(num_classes)
        
    def score(self, X):
        """
        Computes the score matrix: XW + b
        X: (N, D) matrix of input samples
        returns: (N, C) matrix of scores for each class
        """
        return np.dot(X, self.W) + self.b
        
    def softmax(self, scores):
        """
        Computes softmax probabilities for each score
        scores: (N, C) matrix
        returns: (N, C) matrix of probabilities
        """
        # Numerical stability: subtract max score from each row
        shifted_scores = scores - np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(shifted_scores)
        probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        return probs
        
    def predict(self, X):
        """
        Predicts the class label for each sample in X
        X: (N, D) matrix
        returns: (N,) vector of predicted labels
        """
        scores = self.score(X)
        probs = self.softmax(scores)
        return np.argmax(probs, axis=1)

In [ ]:
input_dim = all_train_images.shape[1]  # 3072
num_classes = 10

lc = LinearClassifier(input_dim, num_classes)

# Get scores for the first 5 samples
scores = lc.score(all_test_images[:5])
print("Scores for first 5 samples:\n", scores)

# Get softmax probabilities
probs = lc.softmax(scores)
print("\nSoftmax probabilities for first 5 samples:\n", probs)

# Predict   
preds = lc.predict(all_test_images)
accuracy = np.mean(preds == all_test_labels)
print(f"\nInitial Accuracy (with random weights): {accuracy * 100:.2f}%")

Scores for first 5 samples:
 [[ -8962.64077379 -10774.65425038    104.06628432   4810.68235666
    3026.7762748   -8317.97281139   9395.6300716    3731.93296356
   -6187.87045207   1860.46545657]
 [-12215.55514331 -13924.5031347    4072.58477851  13684.91063241
    4135.38770702 -17806.7314564   20391.72246486  14644.27131597
   -7427.314992    11390.1176875 ]
 [-11493.40828743 -13737.98775291  -1934.3100816    7418.7933803
    2886.10893737 -12442.05532694  18594.17353029  11841.25824242
   -2317.12606325   7535.11031805]
 [ -6314.34301876 -15319.52989329  -3930.9084536   10953.86700679
   -6362.53308479 -18089.85529656  17315.23414747  15234.66158877
   -9612.91792977  10876.27110297]
 [ -6043.92128722  -4549.60593969    518.58286987   3964.68068891
   -2418.97625205  -6293.7515006    9437.72575737   6192.33560124
   -6093.89571628   4092.84208919]]

Softmax probabilities for first 5 samples:
 [[0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.